# REHAB24-6 Dataset Analysis

## Overview
This notebook provides exploratory data analysis of the REHAB24-6 rehabilitation exercise dataset. The dataset supports development of computer vision systems for automated physiotherapy assessment.

## Dataset Specification
| Component | Description |
|-----------|-------------|
| `2d_joints/` | 2D skeletal coordinates extracted via pose estimation (26 joints × 2 coords per frame) |
| `3d_joints/` | 3D motion capture ground truth from marker-based system |
| `videos/` | RGB video recordings at 30fps from frontal (c17) and lateral (c18) camera views |
| `Segmentation.csv` | Frame-level annotations including exercise boundaries and correctness labels |

## Objectives
1. Understand the data structure and file organization
2. Analyze class distribution (correct vs incorrect executions)
3. Visualize skeleton topology and joint trajectories
4. Establish data loading pipeline for subsequent model development

In [ ]:
# Standard imports for data manipulation and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path

# Define dataset root path (relative to notebook location)
DATASET_PATH = Path("../dataset")

# Verify dataset accessibility
print(f"Dataset location: {DATASET_PATH.absolute()}")
print(f"Status: {'Available' if DATASET_PATH.exists() else 'NOT FOUND - check path'}")

## 1. Skeletal Joint Configuration

The dataset employs a 26-joint skeleton model following standard motion capture conventions. Each joint represents an anatomical landmark tracked across video frames.

In [ ]:
# Load and parse joint name definitions
with open(DATASET_PATH / "joints_names.txt", 'r') as f:
    joint_names_raw = f.read()
    
print("Joint Index Mapping:")
print("-" * 30)
print(joint_names_raw)

# Extract joint names into list for programmatic access
joint_names = []
for line in joint_names_raw.strip().split('\n'):
    parts = line.split(':')
    if len(parts) == 2:
        joint_names.append(parts[1].strip())
        
print(f"\nTotal joints: {len(joint_names)}")

## 2. Segmentation Metadata Analysis

The `Segmentation.csv` file provides frame-level annotations for each exercise repetition:

| Column | Description |
|--------|-------------|
| `video_id` | Unique identifier for source video (e.g., PM_000) |
| `repetition_number` | Sequential repetition index within video |
| `exercise_id` | Exercise type (1-6) |
| `first_frame` / `last_frame` | Temporal boundaries of repetition |
| `correctness` | Binary label: 1 = correct execution, 0 = incorrect |

In [ ]:
# Load segmentation annotations (semicolon-delimited CSV)
segmentation = pd.read_csv(DATASET_PATH / "Segmentation.csv", sep=';')

print(f"Total annotated repetitions: {len(segmentation)}")
print(f"\nColumn schema: {list(segmentation.columns)}")
print("\nSample entries:")
segmentation.head(10)

In [ ]:
# Analyze class distribution for correctness labels
correct_count = (segmentation['correctness'] == 1).sum()
incorrect_count = (segmentation['correctness'] == 0).sum()
total = len(segmentation)

print("Classification Label Distribution")
print("=" * 40)
print(f"Correct executions:   {correct_count:4d} ({100*correct_count/total:.1f}%)")
print(f"Incorrect executions: {incorrect_count:4d} ({100*incorrect_count/total:.1f}%)")

# Exercise type distribution
print("\nSamples per Exercise Type")
print("=" * 40)
exercise_counts = segmentation['exercise_id'].value_counts().sort_index()
for ex_id, count in exercise_counts.items():
    print(f"Exercise {ex_id}: {count:4d} repetitions")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class balance plot
labels = ['Incorrect (0)', 'Correct (1)']
values = [incorrect_count, correct_count]
colors = ['#e74c3c', '#27ae60']
axes[0].bar(labels, values, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_ylabel('Number of Samples')
axes[0].set_title('Correctness Label Distribution')
for i, v in enumerate(values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Exercise distribution plot
ex_labels = [f"Ex{i}" for i in exercise_counts.index]
axes[1].bar(ex_labels, exercise_counts.values, color='#3498db', edgecolor='black', linewidth=0.5)
axes[1].set_ylabel('Number of Samples')
axes[1].set_title('Exercise Type Distribution')
axes[1].set_xlabel('Exercise ID')

plt.tight_layout()
plt.show()

## 3. 2D Joint Data Structure

Joint coordinate files are stored as NumPy arrays with the following specification:

- **Format**: `.npy` (NumPy binary)
- **Shape**: `(num_frames, 26, 2)` representing (temporal, spatial, coordinate) dimensions
- **Naming**: `{video_id}-{camera}-{fps}fps.npy`
- **Coordinate space**: Pixel coordinates in original video resolution

In [ ]:
# Enumerate available files per exercise folder
print("2D Joint Data Inventory")
print("=" * 40)
for ex_folder in sorted((DATASET_PATH / "2d_joints").iterdir()):
    if ex_folder.is_dir():
        npy_files = list(ex_folder.glob("*.npy"))
        print(f"{ex_folder.name}: {len(npy_files)} files")

# Load sample file for structure inspection
sample_path = DATASET_PATH / "2d_joints" / "Ex1" / "PM_000-c17-30fps.npy"
joints_2d = np.load(sample_path)

print(f"\nSample File Analysis: {sample_path.name}")
print("-" * 40)
print(f"Array shape: {joints_2d.shape}")
print(f"  - Temporal dimension: {joints_2d.shape[0]} frames")
print(f"  - Spatial dimension:  {joints_2d.shape[1]} joints")
print(f"  - Coordinate dimension: {joints_2d.shape[2]} (x, y)")
print(f"Data type: {joints_2d.dtype}")
print(f"Coordinate range: [{joints_2d.min():.1f}, {joints_2d.max():.1f}] pixels")

## 4. Skeleton Visualization

Visualization requires defining the skeletal topology—which joints connect to form anatomical limbs. The connectivity follows the hierarchical kinematic chain from pelvis through spine to extremities.

In [ ]:
# Define skeletal connectivity as joint index pairs
# Topology follows anatomical kinematic chains

SKELETON_CONNECTIONS = [
    # Axial skeleton (spine chain)
    (0, 1), (1, 2), (2, 3), (3, 4), (4, 5),   # Hips -> Spine -> Neck -> Head
    
    # Left upper extremity
    (2, 6), (6, 7), (7, 8), (8, 9), (9, 10),  # Spine1 -> Shoulder -> Elbow -> Hand
    
    # Right upper extremity
    (2, 11), (11, 12), (12, 13), (13, 14), (14, 15),
    
    # Left lower extremity
    (0, 16), (16, 17), (17, 18), (18, 19), (19, 20),  # Hips -> Hip -> Knee -> Foot
    
    # Right lower extremity
    (0, 21), (21, 22), (22, 23), (23, 24), (24, 25),
]


def render_skeleton(joints, ax, title=""):
    """
    Render 2D skeleton on matplotlib axes.
    
    Parameters:
        joints: ndarray of shape (26, 2) containing joint coordinates
        ax: matplotlib axes object
        title: Optional plot title
    """
    # Draw bone segments
    for idx_a, idx_b in SKELETON_CONNECTIONS:
        x_coords = [joints[idx_a, 0], joints[idx_b, 0]]
        y_coords = [joints[idx_a, 1], joints[idx_b, 1]]
        ax.plot(x_coords, y_coords, 'b-', linewidth=2, alpha=0.7)
    
    # Draw joint markers
    ax.scatter(joints[:, 0], joints[:, 1], c='red', s=40, zorder=5, edgecolor='white')
    
    ax.set_title(title)
    ax.invert_yaxis()  # Match image coordinate system (origin at top-left)
    ax.set_aspect('equal')
    ax.axis('off')


# Render skeleton at multiple time points to show movement progression
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
sample_frames = [0, 100, 200, 300]

for ax, frame_idx in zip(axes, sample_frames):
    if frame_idx < len(joints_2d):
        render_skeleton(joints_2d[frame_idx], ax, f"Frame {frame_idx}")

plt.suptitle("Skeleton Visualization: Temporal Progression", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Extracting Individual Repetitions

Each video contains multiple exercise repetitions. The segmentation metadata specifies frame boundaries, enabling extraction of isolated movement sequences for analysis.

In [ ]:
# Extract segmentation data for sample video
video_id = "PM_000"
video_segments = segmentation[segmentation['video_id'] == video_id]

print(f"Segmentation Data for Video: {video_id}")
print("=" * 50)
print(video_segments[['repetition_number', 'first_frame', 'last_frame', 'correctness']].to_string(index=False))

# Extract first repetition as demonstration
first_rep = video_segments.iloc[0]
start_frame = first_rep['first_frame']
end_frame = first_rep['last_frame']
label = first_rep['correctness']

print(f"\nFirst Repetition Extraction:")
print(f"  Frame range: [{start_frame}, {end_frame}]")
print(f"  Duration: {end_frame - start_frame + 1} frames")
print(f"  Label: {'CORRECT' if label == 1 else 'INCORRECT'}")

# Slice joint array for this repetition
rep_joints = joints_2d[start_frame:end_frame + 1]
print(f"  Extracted shape: {rep_joints.shape}")

## 6. Correct vs Incorrect Movement Comparison

Visual comparison between correctly and incorrectly executed exercises. This illustrates the classification target—identifying postural and kinematic differences that distinguish proper from improper technique.

In [ ]:
def load_repetition(video_id, exercise_folder, segmentation_df, rep_idx=0):
    """
    Load joint data for a specific exercise repetition.
    
    Parameters:
        video_id: Source video identifier
        exercise_folder: Exercise folder name (e.g., 'Ex1')
        segmentation_df: DataFrame containing segmentation metadata
        rep_idx: Repetition index within video
        
    Returns:
        tuple: (joint_array, correctness_label) or (None, None) if unavailable
    """
    joint_path = DATASET_PATH / "2d_joints" / exercise_folder / f"{video_id}-c17-30fps.npy"
    
    if not joint_path.exists():
        return None, None
    
    joints = np.load(joint_path)
    video_segs = segmentation_df[segmentation_df['video_id'] == video_id]
    
    if len(video_segs) <= rep_idx:
        return None, None
        
    seg = video_segs.iloc[rep_idx]
    rep_joints = joints[seg['first_frame']:seg['last_frame'] + 1]
    
    return rep_joints, seg['correctness']


# Identify one correct and one incorrect sample from Exercise 1
ex1_segments = segmentation[segmentation['exercise_id'] == 1]
correct_sample = ex1_segments[ex1_segments['correctness'] == 1].iloc[0]
incorrect_sample = ex1_segments[ex1_segments['correctness'] == 0].iloc[0]

print("Selected Samples for Comparison:")
print(f"  Correct:   {correct_sample['video_id']}, Rep {correct_sample['repetition_number']}")
print(f"  Incorrect: {incorrect_sample['video_id']}, Rep {incorrect_sample['repetition_number']}")

In [ ]:
# Load joint data for both samples
correct_joints = np.load(DATASET_PATH / "2d_joints" / "Ex1" / f"{correct_sample['video_id']}-c17-30fps.npy")
incorrect_joints = np.load(DATASET_PATH / "2d_joints" / "Ex1" / f"{incorrect_sample['video_id']}-c17-30fps.npy")

# Select mid-point frame (peak of movement) for each repetition
correct_frame_idx = (correct_sample['first_frame'] + correct_sample['last_frame']) // 2
incorrect_frame_idx = (incorrect_sample['first_frame'] + incorrect_sample['last_frame']) // 2

# Side-by-side visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Correct execution
render_skeleton(correct_joints[correct_frame_idx], axes[0], "CORRECT Execution")
axes[0].set_facecolor('#d4edda')

# Incorrect execution
render_skeleton(incorrect_joints[incorrect_frame_idx], axes[1], "INCORRECT Execution")
axes[1].set_facecolor('#f8d7da')

plt.suptitle("Exercise 1: Correct vs Incorrect Technique Comparison", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nClassification Objective: Train a model to distinguish these execution patterns based on skeletal kinematics.")

## Summary

### Key Findings

| Aspect | Specification |
|--------|---------------|
| Joint representation | 26 anatomical landmarks per frame |
| Coordinate format | (num_frames, 26, 2) pixel coordinates |
| Exercise types | 6 distinct rehabilitation movements |
| Label distribution | Binary correctness classification |
| Temporal segmentation | Frame-level repetition boundaries |

### Implementation Pipeline

1. **Data Loading**: Extract individual repetitions using segmentation boundaries
2. **Normalization**: Apply torso-based scaling for subject-invariant comparison
3. **Feature Engineering**: Compute joint angles, velocities, and spatial relationships
4. **Reference Modeling**: Build statistical templates from correct executions
5. **Classification**: Train discriminative model for correctness prediction

### Next Steps
- Implement skeleton normalization module
- Build reference model from correct samples
- Develop distance-based comparison metrics
- Train classification model with cross-validation